In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets transformers accelerate soundfile librosa evaluate jiwer tensorboard gradio python-Levenshtein

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
from datasets import load_dataset, DatasetDict, Audio
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor
import evaluate


In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

# Încărcare și filtrare inițială
vreme = DatasetDict({
    "train": load_dataset("iulik-pisik/audio_vreme", split='train', trust_remote_code=True).filter(lambda x: x['gender'] == 'masc'),
    "validation": load_dataset("iulik-pisik/audio_vreme", split='validation', trust_remote_code=True).filter(lambda x: x['gender'] == 'masc'),
    "test": load_dataset("iulik-pisik/audio_vreme", split='test', trust_remote_code=True).filter(lambda x: x['gender'] == 'masc')
})


# Divizare și realocare
additional_train = vreme["validation"].filter(lambda x: x['name'] == "Florin Busuioc")
additional_test = vreme["validation"].filter(lambda x: x['name'] == "Cosmin Stan")

vreme["train"] = concatenate_datasets([vreme["train"], additional_train])
vreme["test"] = concatenate_datasets([vreme["test"], additional_test])


horoscop = DatasetDict({
    "train": load_dataset("iulik-pisik/horoscop_neti", split='train+validation', trust_remote_code=True),
    "test": load_dataset("iulik-pisik/horoscop_neti", split='test', trust_remote_code=True)
})


# all_data = DatasetDict({
#     "train": concatenate_datasets([vreme["train"], horoscop["train"]]),
#     "test": concatenate_datasets([vreme["test"], horoscop["test"]])
# })


Generating train split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 0it [00:00, ?it/s]
Se citesc datele...: 1000it [00:00, 9206.37it/s]
Se citesc datele...: 2242it [00:00, 10532.56it/s]


Generating test split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 578it [00:00, 11773.80it/s]


Generating validation split: 0 examples [00:00, ? examples/s]

...


Se citesc datele...: 1276it [00:00, 14081.51it/s]


Generating test split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 184it [00:00, 11760.03it/s]


Generating validation split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 364it [00:00, 13539.01it/s]


In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Romanian", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="smallnian", task="transcribe")


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
vreme = vreme.map(prepare_dataset, remove_columns=vreme.column_names["train"], num_proc=4)
horoscop = horoscop.map(prepare_dataset, remove_columns=horoscop.column_names["train"], num_proc=4)


/usr/local/lib/python3.10/dist-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Map (num_proc=4):   0%|          | 0/2518 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/588 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/425 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1640 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/184 [00:00<?, ? examples/s]

In [ ]:
del vreme['validation']

In [ ]:
vreme

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2518
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 425
    })
})

In [ ]:
horoscop

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1640
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 184
    })
})

In [ ]:
# Combina seturile de date preprocesate
all_data = DatasetDict({
    "train": concatenate_datasets([vreme["train"], horoscop["train"]]),
    "test": concatenate_datasets([vreme["test"], horoscop["test"]])
})


In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.generation_config.language = "ro"  # define your language of choice here


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./all_data_model_small",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=all_data["train"],
    eval_dataset=all_data["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
1000,0.033200,0.113458,9.194474
2000,0.003500,0.149135,10.844995
3000,0.000500,0.166582,8.467887
4000,0.000400,0.170841,8.512738


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/do

TrainOutput(global_step=4000, training_loss=0.07975976229365915, metrics={'train_runtime': 11199.67, 'train_samples_per_second': 5.714, 'train_steps_per_second': 0.357, 'total_flos': 1.84608080584704e+19, 'train_loss': 0.07975976229365915, 'epoch': 15.38})

In [ ]:
kwargs = {
    "dataset_tags": "iulik-pisik/horoscop_neti",
    "dataset": "Vreme ProTv  Horoscop Neti",
    "dataset_args": "config: ro, split: test",
    "language": "ro",
    "model_name": "Whisper Small Romanian - Vreme & Horoscop",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
    "tags": "hf-asr-leaderboard",
}


In [ ]:
trainer.push_to_hub(**kwargs)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


BadRequestError:  (Request ID: Root=1-661eefd3-7c21079d2521531931ac7d2a;bd969b9f-bf1d-40a8-a344-488215b9c362)

Bad request for commit endpoint:
"model-index[0].results[0].dataset.config" must be a string

In [ ]:
# tokenizer.push_to_hub("username/model-id")